# Intel® Extension for Scikit-learn Ridge Regression for New York City Bike Share dataset

In [1]:
from timeit import default_timer as timer
from sklearn import metrics
from sklearn.model_selection import train_test_split
import warnings
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder
from IPython.display import HTML

warnings.filterwarnings("ignore")

### Download the data

In [2]:
dataset = fetch_openml(data_id=43526, as_frame=True)

In [3]:
# Check the keys of the dataset to understand its structure
print(dataset.keys())

dict_keys(['data', 'target', 'frame', 'categories', 'feature_names', 'target_names', 'DESCR', 'details', 'url'])


In [14]:
import pandas as pd

# Access the data as a DataFrame
data = dataset.frame

# Convert date columns to datetime
data['Start_Time'] = pd.to_datetime(data['Start_Time'])
data['Stop_Time'] = pd.to_datetime(data['Stop_Time'])

# Extract useful features from datetime columns
data['Start_Year'] = data['Start_Time'].dt.year
data['Start_Month'] = data['Start_Time'].dt.month
data['Start_Day'] = data['Start_Time'].dt.day
data['Start_Hour'] = data['Start_Time'].dt.hour

data['Stop_Year'] = data['Stop_Time'].dt.year
data['Stop_Month'] = data['Stop_Time'].dt.month
data['Stop_Day'] = data['Stop_Time'].dt.day
data['Stop_Hour'] = data['Stop_Time'].dt.hour

# Drop the original datetime columns
data = data.drop(columns=['Start_Time', 'Stop_Time'])

# Encode categorical variables
for col in ['Start_Station_Name', 'End_Station_Name', 'Gender', 'User_Type']:
    le = LabelEncoder().fit(data[col])
    data[col] = le.transform(data[col])

# Set the target variable
data['target'] = data['Trip_Duration']

# Separate features and target
x = data.drop(columns=['target', 'Trip_Duration'])
y = data['target']

# Select categorical columns based on data types
categorical_cols = x.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical columns based on data types: {categorical_cols}')

# Select columns with a small number of unique values
low_cardinality_cols = [col for col in x.columns if x[col].nunique() < 10]
print(f'Columns with low cardinality: {low_cardinality_cols}')

# Combine both methods to get a comprehensive list of categorical columns
categorical_cols = list(set(categorical_cols + low_cardinality_cols))
print(f'Final list of categorical columns: {categorical_cols}')

Categorical columns based on data types: []
Columns with low cardinality: ['User_Type', 'Gender', 'Start_Year', 'Stop_Year']
Final list of categorical columns: ['Gender', 'User_Type', 'Stop_Year', 'Start_Year']


### Preprocessing
Let's encode categorical features with LabelEncoder

In [15]:
# Ensure x and y are defined and not None
if x is not None and y is not None:
    for col in ['User_Type', 'Gender']:
        if col in x.columns:
            le = LabelEncoder().fit(x[col])
            x[col] = le.transform(x[col])
        else:
            print(f"Column {col} does not exist in the DataFrame.")

    # Split the data
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=0)
    print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)
else:
    print("x or y is None. Please check your data.")

(661951, 22) (73551, 22) (661951,) (73551,)


In [16]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

scaler_x = MinMaxScaler()
scaler_y = StandardScaler()

In [17]:
y_train = y_train.to_numpy().reshape(-1, 1)
y_test = y_test.to_numpy().reshape(-1, 1)

scaler_x.fit(x_train)
x_train = scaler_x.transform(x_train)
x_test = scaler_x.transform(x_test)

scaler_y.fit(y_train)
y_train = scaler_y.transform(y_train).ravel()
y_test = scaler_y.transform(y_test).ravel()

In [18]:
from sklearnex import patch_sklearn

patch_sklearn()

Intel(R) Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [19]:
from sklearn.linear_model import Ridge

params = {
    "alpha": 0.3,
    "fit_intercept": False,
    "random_state": 0,
    "copy_X": False,
}
start = timer()
model = Ridge(random_state=0).fit(x_train, y_train)
train_patched = timer() - start
f"Intel® extension for Scikit-learn time: {train_patched:.2f} s"

'Intel® extension for Scikit-learn time: 0.04 s'

In [20]:
y_predict = model.predict(x_test)
mse_metric_opt = metrics.mean_squared_error(y_test, y_predict)
f"Patched Scikit-learn MSE: {mse_metric_opt}"

'Patched Scikit-learn MSE: 0.29078674972552815'

In [21]:
from sklearnex import unpatch_sklearn

unpatch_sklearn()

In [22]:
from sklearn.linear_model import Ridge

start = timer()
model = Ridge(random_state=0).fit(x_train, y_train)
train_unpatched = timer() - start
f"Original Scikit-learn time: {train_unpatched:.2f} s"

'Original Scikit-learn time: 0.13 s'

In [23]:
y_predict = model.predict(x_test)
mse_metric_original = metrics.mean_squared_error(y_test, y_predict)
f"Original Scikit-learn MSE: {mse_metric_original}"

'Original Scikit-learn MSE: 0.29078674972650354'

In [24]:
HTML(
    f"<h3>Compare MSE metric of patched Scikit-learn and original</h3>"
    f"MSE metric of patched Scikit-learn: {mse_metric_opt} <br>"
    f"MSE metric of unpatched Scikit-learn: {mse_metric_original} <br>"
    f"Metrics ratio: {mse_metric_opt/mse_metric_original} <br>"
    f"<h3>With Scikit-learn-intelex patching you can:</h3>"
    f"<ul>"
    f"<li>Use your Scikit-learn code for training and prediction with minimal changes (a couple of lines of code);</li>"
    f"<li>Fast execution training and prediction of Scikit-learn models;</li>"
    f"<li>Get the similar quality</li>"
    f"<li>Get speedup in <strong>{(train_unpatched/train_patched):.1f}</strong> times.</li>"
    f"</ul>"
)